In [3]:
%pip install seaborn


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance

In [6]:
file_path = "CSI_7_MAL_2324_CW_resit_data.xlsx"

diagnosis_df = pd.read_excel(
    file_path,
    sheet_name="Diagnosis target"
)

cognitive_scores_df = pd.read_excel(
    file_path,
    sheet_name="Cognitive score"
)

data_df = pd.read_excel(
    file_path,
    sheet_name="Data"
)

print("Diagnosis data:", diagnosis_df.shape)
print("Cognitive data:", cognitive_scores_df.shape)
print("Patient data:", data_df.shape)

Diagnosis data: (1737, 4)
Cognitive data: (1737, 10)
Patient data: (1737, 361)


In [7]:
print("Diagnosis columns:")
print(diagnosis_df.columns.tolist())

print("\nCognitive score columns:")
print(cognitive_scores_df.columns.tolist())

print("\nPatient data columns:")
print(data_df.columns.tolist())

display(diagnosis_df.head())
display(cognitive_scores_df.head())
display(data_df.head())

Diagnosis columns:
['RID', 'Test_data', 'Diagnosis', 'FDG_PET']

Cognitive score columns:
['RID', 'Test_data', 'CDRSB', 'ADAS11', 'ADAS13', 'MMSE', 'RAVLT_immediate', 'RAVLT_learning', 'RAVLT_forgetting', 'RAVLT_perc_forgetting']

Patient data columns:
['RID', 'Test_data', 'Age', 'Gender', 'Year_education', 'Ethnicity', 'Race', 'Marital_status', 'High_risk_ApoE4', 'Ventricles', 'Hippocampus', 'WholeBrain', 'Entorhinal', 'Fusiform', 'MidTemp', 'Intra cranial volume', 'Volume (WM Parcellation) of RightPallidum', 'Volume (Cortical Parcellation) of RightParacentral', 'Surface Area of RightParacentral', 'Cortical Thickness Average of RightParacentral', 'Cortical Thickness Standard Deviation of RightParacentral', 'Volume (Cortical Parcellation) of RightParahippocampal', 'Surface Area of RightParahippocampal', 'Cortical Thickness Average of RightParahippocampal', 'Cortical Thickness Standard Deviation of RightParahippocampal', 'Volume (Cortical Parcellation) of RightParsOpercularis', 'Surface

,RID,Test_data,Diagnosis,FDG_PET
0,2,0,CN,1.36926
1,3,0,AD,1.09079
2,4,1,LMCI,NaN
3,5,0,CN,1.29799
4,6,0,LMCI,NaN


,RID,Test_data,CDRSB,ADAS11,ADAS13,MMSE,RAVLT_immediate,RAVLT_learning,RAVLT_forgetting,RAVLT_perc_forgetting
0,2,0,0.0,10.67,18.67,28,44.0,4.0,6.0,54.5455
1,3,0,4.5,22.00,31.00,20,22.0,1.0,4.0,100.0000
2,4,1,1.0,14.33,21.33,27,37.0,7.0,4.0,36.3636
3,5,0,0.0,8.67,14.67,29,37.0,4.0,4.0,44.4444
4,6,0,0.5,18.67,25.67,25,30.0,1.0,5.0,83.3333


,RID,Test_data,Age,Gender,Year_education,Ethnicity,Race,Marital_status,High_risk_ApoE4,Ventricles,Hippocampus,WholeBrain,Entorhinal,Fusiform,MidTemp,Intra cranial volume,Volume (WM Parcellation) of RightPallidum,Volume (Cortical Parcellation) of RightParacentral,Surface Area of RightParacentral,Cortical Thickness Average of RightParacentral,Cortical Thickness Standard Deviation of RightParacentral,Volume (Cortical Parcellation) of RightParahippocampal,Surface Area of RightParahippocampal,Cortical Thickness Average of RightParahippocampal,Cortical Thickness Standard Deviation of RightParahippocampal,Volume (Cortical Parcellation) of RightParsOpercularis,Surface Area of RightParsOpercularis,Cortical Thickness Average of RightParsOpercularis,Cortical Thickness Standard Deviation of RightParsOpercularis,Volume (Cortical Parcellation) of RightParsOrbitalis,Surface Area of RightParsOrbitalis,Cortical Thickness Average of RightParsOrbitalis,Cortical Thickness Standard Deviation of RightParsOrbitalis,Volume (Cortical Parcellation) of RightParsTriangularis,Surface Area of RightParsTriangularis,Cortical Thickness Average of RightParsTriangularis,Cortical Thickness Standard Deviation of RightParsTriangularis,Volume (Cortical Parcellation) of RightPericalcarine,Surface Area of RightPericalcarine,Cortical Thickness Average of RightPericalcarine,...,Surface Area of RightHemisphere,Volume (Cortical Parcellation) of RightHemisphereWM,Volume (WM Parcellation) of RightHippocampus,Volume (WM Parcellation) of RightInferiorLateralVentricle,Volume (WM Parcellation) of FifthVentricle,Volume (Cortical Parcellation) of RightInferiorParietal,Surface Area of RightInferiorParietal,Cortical Thickness Average of RightInferiorParietal,Cortical Thickness Standard Deviation of RightInferiorParietal,Volume (Cortical Parcellation) of RightInferiorTemporal,Surface Area of RightInferiorTemporal,Cortical Thickness Average of RightInferiorTemporal,Cortical Thickness Standard Deviation of RightInferiorTemporal,Volume (WM Parcellation) of RightInterior,Volume (Cortical Parcellation) of RightIsthmusCingulate,Surface Area of RightIsthmusCingulate,Cortical Thickness Average of RightIsthmusCingulate,Cortical Thickness Standard Deviation of RightIsthmusCingulate,Volume (Cortical Parcellation) of RightLateralOccipital,Surface Area of RightLateralOccipital,Cortical Thickness Average of RightLateralOccipital,Cortical Thickness Standard Deviation of RightLateralOccipital,Volume (Cortical Parcellation) of RightLateralOrbitofrontal,Surface Area of RightLateralOrbitofrontal,Cortical Thickness Average of RightLateralOrbitofrontal,Cortical Thickness Standard Deviation of RightLateralOrbitofrontal,Volume (WM Parcellation) of RightLateralVentricle,Volume (Cortical Parcellation) of RightLingual,Surface Area of RightLingual,Cortical Thickness Average of RightLingual,Cortical Thickness Standard Deviation of RightLingual,Volume (Cortical Parcellation) of RightMedialOrbitofrontal,Surface Area of RightMedialOrbitofrontal,Cortical Thickness Average of RightMedialOrbitofrontal,Cortical Thickness Standard Deviation of RightMedialOrbitofrontal,Volume (Cortical Parcellation) of RightMiddleTemporal,Surface Area of RightMiddleTemporal,Cortical Thickness Average of RightMiddleTemporal,Cortical Thickness Standard Deviation of RightMiddleTemporal,Volume (WM Parcellation) of FourthVentricle
0,2,0,74.3,Male,16,Not Hisp/Latino,White,Married,0.0,118233.0,8336.0,1229740.0,4177.0,16559.0,27936.0,1984660.0,1602,4953,1947,2.332,0.738,2443,652,3.19,0.833,3466,1303,2.462,0.61,1963,695,2.301,0.671,2809,1147,2.141,0.629,2293,1700,1.529,...,101251,,4219,2436,24,15789,6612,2.243,0.65,11155,3273,2.901,0.688,,2439,898,2.429,0.639,14564,6216,2.085,0.699,6795,2649,2.344,0.842,55425,8142,3516,2.079,0.694,3835,1622,2.077,0.746,15683,4272,3.028,0.649,4396
1,3,0,81.3,Male,18,Not Hisp/Latino,White,Married,1.0,84599.0,5319.0,1129830.0,1791.0,15506.0,18422.0,1920690.0,1692,3539,1579,2.121,0.645,1881,697,2.214,0.857,3463,13

In [8]:
print(diagnosis_df["Diagnosis"].value_counts(dropna=False))

Diagnosis
LMCI    562
CN      417
AD      342
EMCI    310
SMC     106
Name: count, dtype: int64


# Handling missing values

In [9]:
# Check whether RID is unique in each sheet
for name, df in {
    "Diagnosis": diagnosis_df,
    "Cognitive": cognitive_scores_df,
    "Patient data": data_df
}.items():

    print(
        name,
        "duplicate RID count:",
        df["RID"].duplicated().sum()
    )

Diagnosis duplicate RID count: 0
Cognitive duplicate RID count: 0
Patient data duplicate RID count: 0


In [12]:
# Keep the diagnosis and target information
diagnosis_clean = diagnosis_df[
    ["RID", "Diagnosis", "FDG_PET"]
].copy()


# Remove split indicators from predictor sheets
cognitive_clean = cognitive_scores_df.drop(
    columns=["Test_data"],
    errors="ignore"
).copy()


data_clean = data_df.drop(
    columns=["Test_data"],
    errors="ignore"
).copy()

# Check unique identifiers before merging
assert diagnosis_clean["RID"].is_unique
assert cognitive_clean["RID"].is_unique
assert data_clean["RID"].is_unique

# Merge the datasets
merged_df = pd.merge(
    diagnosis_clean,
    cognitive_clean,
    on="RID",
    how="inner",
    validate="one_to_one"
)

merged_df = pd.merge(
    merged_df,
    data_clean,
    on="RID",
    how="inner",
    validate="one_to_one"
)
print("Merged dataset shape:", merged_df.shape)

display(merged_df.head())


Merged dataset shape: (1737, 370)


,RID,Diagnosis,FDG_PET,CDRSB,ADAS11,ADAS13,MMSE,RAVLT_immediate,RAVLT_learning,RAVLT_forgetting,RAVLT_perc_forgetting,Age,Gender,Year_education,Ethnicity,Race,Marital_status,High_risk_ApoE4,Ventricles,Hippocampus,WholeBrain,Entorhinal,Fusiform,MidTemp,Intra cranial volume,Volume (WM Parcellation) of RightPallidum,Volume (Cortical Parcellation) of RightParacentral,Surface Area of RightParacentral,Cortical Thickness Average of RightParacentral,Cortical Thickness Standard Deviation of RightParacentral,Volume (Cortical Parcellation) of RightParahippocampal,Surface Area of RightParahippocampal,Cortical Thickness Average of RightParahippocampal,Cortical Thickness Standard Deviation of RightParahippocampal,Volume (Cortical Parcellation) of RightParsOpercularis,Surface Area of RightParsOpercularis,Cortical Thickness Average of RightParsOpercularis,Cortical Thickness Standard Deviation of RightParsOpercularis,Volume (Cortical Parcellation) of RightParsOrbitalis,Surface Area of RightParsOrbitalis,...,Surface Area of RightHemisphere,Volume (Cortical Parcellation) of RightHemisphereWM,Volume (WM Parcellation) of RightHippocampus,Volume (WM Parcellation) of RightInferiorLateralVentricle,Volume (WM Parcellation) of FifthVentricle,Volume (Cortical Parcellation) of RightInferiorParietal,Surface Area of RightInferiorParietal,Cortical Thickness Average of RightInferiorParietal,Cortical Thickness Standard Deviation of RightInferiorParietal,Volume (Cortical Parcellation) of RightInferiorTemporal,Surface Area of RightInferiorTemporal,Cortical Thickness Average of RightInferiorTemporal,Cortical Thickness Standard Deviation of RightInferiorTemporal,Volume (WM Parcellation) of RightInterior,Volume (Cortical Parcellation) of RightIsthmusCingulate,Surface Area of RightIsthmusCingulate,Cortical Thickness Average of RightIsthmusCingulate,Cortical Thickness Standard Deviation of RightIsthmusCingulate,Volume (Cortical Parcellation) of RightLateralOccipital,Surface Area of RightLateralOccipital,Cortical Thickness Average of RightLateralOccipital,Cortical Thickness Standard Deviation of RightLateralOccipital,Volume (Cortical Parcellation) of RightLateralOrbitofrontal,Surface Area of RightLateralOrbitofrontal,Cortical Thickness Average of RightLateralOrbitofrontal,Cortical Thickness Standard Deviation of RightLateralOrbitofrontal,Volume (WM Parcellation) of RightLateralVentricle,Volume (Cortical Parcellation) of RightLingual,Surface Area of RightLingual,Cortical Thickness Average of RightLingual,Cortical Thickness Standard Deviation of RightLingual,Volume (Cortical Parcellation) of RightMedialOrbitofrontal,Surface Area of RightMedialOrbitofrontal,Cortical Thickness Average of RightMedialOrbitofrontal,Cortical Thickness Standard Deviation of RightMedialOrbitofrontal,Volume (Cortical Parcellation) of RightMiddleTemporal,Surface Area of RightMiddleTemporal,Cortical Thickness Average of RightMiddleTemporal,Cortical Thickness Standard Deviation of RightMiddleTemporal,Volume (WM Parcellation) of FourthVentricle
0,2,CN,1.36926,0.0,10.67,18.67,28,44.0,4.0,6.0,54.5455,74.3,Male,16,Not Hisp/Latino,White,Married,0.0,118233.0,8336.0,1229740.0,4177.0,16559.0,27936.0,1984660.0,1602,4953,1947,2.332,0.738,2443,652,3.19,0.833,3466,1303,2.462,0.61,1963,695,...,101251,,4219,2436,24,15789,6612,2.243,0.65,11155,3273,2.901,0.688,,2439,898,2.429,0.639,14564,6216,2.085,0.699,6795,2649,2.344,0.842,55425,8142,3516,2.079,0.694,3835,1622,2.077,0.746,15683,4272,3.028,0.649,4396
1,3,AD,1.09079,4.5,22.00,31.00,20,22.0,1.0,4.0,100.0000,81.3,Male,18,Not Hisp/Latino,White,Married,1.0,84599.0,5319.0,1129830.0,1791.0,15506.0,18422.0,1920690.0,1692,3539,1579,2.121,0.645,1881,697,2.214,0.857,3463,1350,2.295,0.648,1864,685,...,92965.4,,2376,5921,93,11917,4849,2.19,0.647,8355,2910,2.377,0.804,,2204,938,2.052,0.709,12672,5666,1.994,0.623,6030,2543,2.173,0.767,28743,6254,3130,1.765,0.591,3681,1734,1.942,0.696,10387,3316,2.545,0.686,3304
2,4,LMCI,NaN,1.0,14.33,21.33,27,37.0,7.0,4.0,36.3636,67.5,Male,1

In [14]:
missing_values = merged_df.isnull().sum()

missing_summary = pd.DataFrame({
    "Missing values": missing_values,
    "Missing percentage": (
        missing_values / len(merged_df)
    ) * 100
})

missing_summary = missing_summary.sort_values(
    by="Missing percentage",
    ascending=False
)

display(missing_summary.head(20))
print("Total missing values:", merged_df.isnull().sum().sum())

,Missing values,Missing percentage
FDG_PET,434,24.985607
Volume (WM Parcellation) of FifthVentricle,291,16.753022
Fusiform,271,15.601612
MidTemp,271,15.601612
Entorhinal,271,15.601612
Hippocampus,248,14.277490
Ventricles,82,4.720783
WholeBrain,48,2.763385
Intra cranial volume,15,0.863558
ADAS13,14,0.805987


Total missing values: 2003


In [16]:
# Remove records without a target label
merged_df = merged_df.dropna(
    subset=["Diagnosis"]
).copy()

# Clean whitespace
merged_df["Diagnosis"] = (
    merged_df["Diagnosis"].astype(str).str.strip()
)

# Check diagnosis distribution
print(merged_df["Diagnosis"].value_counts())

# Map all five diagnosis categories
category_mapping = {
    "CN": "Cognitively Normal",
    "AD": "Alzheimer's Disease",
    "LMCI": "Late Mild Cognitive Impairment",
    "EMCI": "Early Mild Cognitive Impairment",
    "SMC": "Significant Memory Concern"
}

# Check for unexpected labels
unknown_labels = (
    set(merged_df["Diagnosis"].unique())
    - set(category_mapping.keys())
)

if unknown_labels:
    raise ValueError(
        f"Unexpected diagnosis labels: {unknown_labels}"
    )

# Convert abbreviations into readable labels
merged_df["Diagnosis"] = (
    merged_df["Diagnosis"].map(category_mapping)
)

print("\nUpdated diagnosis distribution:")
print(merged_df["Diagnosis"].value_counts())

Diagnosis
LMCI    562
CN      417
AD      342
EMCI    310
SMC     106
Name: count, dtype: int64

Updated diagnosis distribution:
Diagnosis
Late Mild Cognitive Impairment     562
Cognitively Normal                 417
Alzheimer's Disease                342
Early Mild Cognitive Impairment    310
Significant Memory Concern         106
Name: count, dtype: int64


In [19]:
# Target variable
y = merged_df["Diagnosis"]

# Input features
X = merged_df.drop(
    columns=["Diagnosis", "RID"]
)

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

Feature shape: (1737, 368)
Target shape: (1737,)

Target distribution:
Diagnosis
Late Mild Cognitive Impairment     562
Cognitively Normal                 417
Alzheimer's Disease                342
Early Mild Cognitive Impairment    310
Significant Memory Concern         106
Name: count, dtype: int64
